# Canny + Clahe + Thres
## Aplicación de técnicas de canny, clahe y thres en imágenes dentales
## Silvia Evangelina Mendez Varela
## Procesamiento de imágenes dentales mediante canny, clahe y threshold
## Objetivo: Transformar una imágen compleja en una representación mác lcara, donde las estructuras importantes puedan identificarse fácilmente
## Entrada: Imagenes dentales con sarro posteriores y frontales
## Salida: Imagenes dentales con sarro posteriores y frontales con aplicación de técnicas canny, clahe y threshold en tres canales 
## Dependencias principales: OpenCV, Numpy, Tkinter, Os

In [1]:
conda env export > entorno_imagenes.yml


Note: you may need to restart the kernel to use updated packages.


In [2]:
#1.Importaciones
import os  # Librería para manejar archivos y carpetas
import cv2  # OpenCV para procesamiento de imágenes
import numpy as np  # Librería para manejar matrices (imágenes)
from tkinter import Tk, filedialog  # Para abrir ventanas y seleccionar carpetas

# Crear ventana oculta (no se muestra interfaz principal)
root = Tk()
root.withdraw()  # Oculta la ventana principal de tkinter

# Abrir ventana para seleccionar carpeta de entrada
carpeta_imagenes = filedialog.askdirectory(title="Selecciona la carpeta de imágenes")

#  Abrir ventana para seleccionar carpeta de salida
carpeta_salida = filedialog.askdirectory(title="Selecciona la carpeta de salida")

# Verificar que el usuario seleccionó ambas carpetas
if not carpeta_imagenes or not carpeta_salida:
    print(" No seleccionaste una carpeta.")  # Mensaje de error
    exit()  # Termina el programa

# Crear carpeta de salida si no existe
if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)  # Crea la carpeta automáticamente

#2.Parámetros de configuración 
# Crear objeto CLAHE (mejora de contraste)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))  
# clipLimit controla el contraste máximo
# tileGridSize divide la imagen en bloques para mejorar contraste local

# Parámetros para detección de bordes (Canny)
lower_threshold = 30  # Umbral inferior
upper_threshold = 70  # Umbral superior

# Parámetros para umbralización (threshold)
umbral_min = 100  # Valor mínimo de intensidad
umbral_max = 255  # Valor máximo (blanco)

# Formatos de imagen permitidos
formatos_validos = (".jpg", ".png", ".jpeg","webp")

#3.Funciones
# Recorrer todos los archivos en la carpeta seleccionada
for archivo in os.listdir(carpeta_imagenes):

    # Filtrar solo archivos de imagen
    if archivo.lower().endswith(formatos_validos):

        # Construir la ruta completa de la imagen
        ruta = os.path.join(carpeta_imagenes, archivo)

        # Leer la imagen en color (BGR)
        imagen = cv2.imread(ruta)

        # Verificar que la imagen se cargó correctamente
        if imagen is None:
            continue  # Si falla, pasa a la siguiente imagen

        # Separar la imagen en canales azul, verde y rojo
        b, g, r = cv2.split(imagen)

        # Guardar los canales en un diccionario para procesarlos fácilmente
        canales = {'Azul': b, 'Verde': g, 'Rojo': r}

        # Lista donde se guardarán los resultados procesados
        resultados = []

#4.Ejecución principal
        # Procesar cada canal
        for nombre, canal in canales.items():

            # Aplicar CLAHE (mejora de contraste)
            clahe_img = clahe.apply(canal)

            # Aplicar umbralización (binarizar imagen)
            _, umbral = cv2.threshold(clahe_img, umbral_min, umbral_max, cv2.THRESH_BINARY)

            #Detectar bordes con Canny
            edges = cv2.Canny(umbral, lower_threshold, upper_threshold)

            # Convertir imágenes de gris a BGR para poder combinarlas
            clahe_bgr = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR)
            umbral_bgr = cv2.cvtColor(umbral, cv2.COLOR_GRAY2BGR)
            edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

            # Agregar texto indicando el tipo de procesamiento
            cv2.putText(clahe_bgr, f'CLAHE {nombre}', (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255),1)

            cv2.putText(umbral_bgr, f'Threshold {nombre}', (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255),1)

            cv2.putText(edges_bgr, f'Canny {nombre}', (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255),1)

            # Guardar los resultados en la lista
            resultados.extend([clahe_bgr, umbral_bgr, edges_bgr])

        # Agregar título a la imagen original
        cv2.putText(imagen, 'Original', (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255),1)

        # Combinar imagen original + todos los resultados horizontalmente
        final = np.hstack([imagen] + resultados)

        # Definir ruta de salida
        salida = os.path.join(carpeta_salida, f"resultado_{archivo}")

        #Guardar la imagen final
        cv2.imwrite(salida, final)

        #Mostrar mensaje de éxito
        print(f"Guardado: {archivo}")

# Mensaje final
print("Proceso terminado")

Guardado: images.jpeg
Guardado: sarro.jpg
Guardado: sf.jpg
Guardado: sp.png
Guardado: sf5.jpg
Guardado: sf4.jpg
Guardado: sarro-dental-supragingival.jpg
Guardado: sarro-dientes.jpg
Guardado: sarrodental.webp
Guardado: maxillary-front-teeth-wedgeshaped-cervical-260nw-2736981513.webp
Proceso terminado
